# Training and Evaluating Neural Networks on California Housing

### PyTorch - Build, Train and Evalaute Regression Neural Nets
**Géron supplementary learning/practice**


### FLOW:
**Build the model**
* model = nn.Linear(8, 1)

**Choose how to measure errors and update the model**
* mse = nn.MSELoss()  - Measure errors and update
* optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  - The optimizer controls how the model’s weights and biases are updated. The learning rate scales those updates.

**Train**
* optimizer.zero_grad() - Clear gradients left from an earlier update
* y_pred = model(X_batch) - Make predictions using the current weights
* loss = mse(y_pred, y_batch) - Measure the prediction errors
* loss.backward() - Calculate gradients: how each weight and bias affects the loss.
* optimizer.step() - Use those gradients to update the weights and biases.

This weight update is the learning. One pass through all training batches is an epoch. With 20 epochs, the pass gets repeated 20 times.

**Evaluate:**

model.eval()                            #selects evaluation behavior for layers that behave differently during training.
with torch.no_grad():                   #skips gradient calculations.
    y_pred = model(X_valid)
    valid_mse = mse(y_pred, y_valid)
print("Validation MSE:", valid_mse.item())


**Batch GD: one update per epoch**
y_pred = model(X_train)

**Mini-batch GD: many updates per epoch**
for X_batch, y_batch in train_loader:
    y_pred = model(X_batch)     

In [1]:
import torch

Goal: Creating tensor, running back propagation, computing the gradients of f with regard to x; forward pass + backward pass + gradient descent step

In [2]:
x = torch.tensor(5.0, requires_grad=True)  #tensor creation = 5, and this is variable
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

grad_fn=<PowBackward0> --> tells the PyTorch how to backpropagate the gradients through this operation

In [3]:
f.backward()   #Calulating gradient 
x.grad         #Show derivative for f (=2x -> at 5 = 10), derivative of f with regard to x

tensor(10.)

In [4]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x ** 2  # forward pass
    f.backward()  # backward pass
    with torch.no_grad():
        x -= learning_rate * x.grad  # gradient descent step

    x.grad.zero_()  # reset the gradients

# Implementing Linear Regression - California Housing (Gerone)

In [5]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

X = housing.data # Housing features
y = housing.target # Target house value

print(X.shape)
print(y.shape)

(20640, 8)
(20640,)


Splitting data for training, validation and test.
* training - learn weights and biases
* Validation - compare models and chooses settings
* test - final assessment after my choices are finished

In [6]:
#Split images for trainig and testing: 80/20

from sklearn.model_selection import train_test_split

#Reserve 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Split the remainig 80% inot 60% training and 20% validation
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)

print('Training:', X_train.shape, y_train.shape)
print('Valid:', X_valid.shape, y_valid.shape)
print('Test:', X_test.shape, y_test.shape)

Training: (15480, 8) (15480,)
Valid: (5160, 8) (5160,)
Test: (4128, 8) (4128,)


In [7]:
#Converting to tensor and normalising the features (same scale across all features)

X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)

means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdim=True)

X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [8]:
#Converting targets to tensors (Resahping column vectors to 2 dimensional from 1-dimensional NumPy)

y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

# Manual Linear Regression
Weight parameters created: 
- w - a column vector with one weight per input dimension
- b - bias parameter 

Process: Predict -> Measure error -> Calulate Gradient -> Update

In [9]:
#Create/ Initialise Parameteres of Linear regression - weight and bias

torch.manual_seed(42)
n_features = X_train.shape[1]  # there are 8 input features
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

### Trainig model
- using autodiff to compute the gradients
- using batch gradient descent (BGD) with the full training set at each training step

In [10]:
print("Prediction:", (X_train @ w + b).shape)
print("Target:", y_train.shape)

Prediction: torch.Size([15480, 1])
Target: torch.Size([15480, 1])


* Learnign rate - defines how big a step the model takes when updating thw weights, If too small - slow learning process, if too large - updates can get uncontroled, cosuign loss or increase.

    <b>w -= learning_rate * w.grad</b> 
- w.grad determines the update direction and contributes to its size
- Learnign rate - scales the update. A higher learning rate doesn’t always mean faster learning.

In [11]:
#Training model

learning_rate = 0.4                             #how big a step the model takes when updating the weight 
n_epochs = 20                                   #20 time running thriugh the trainign set, no stop 
for epoch in range(n_epochs):
    y_pred = X_train @ w + b                    #forward pass and the mean squared error loss
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()                             #computing the gradients of the loss regarding every model params (via autograd)
    with torch.no_grad():                       #PyTorch consumes less RAM and run faster
        b -= learning_rate * b.grad             #gradient descent step
        w -= learning_rate * w.grad             
        b.grad.zero_()                          #gradiengt reset to zero 
        w.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}") #print epoch and loss

Epoch 1/20, Loss: 16.01335906982422
Epoch 2/20, Loss: 4.675838947296143
Epoch 3/20, Loss: 2.1238961219787598
Epoch 4/20, Loss: 1.2568167448043823
Epoch 5/20, Loss: 0.928217351436615
Epoch 6/20, Loss: 0.7923635244369507
Epoch 7/20, Loss: 0.7286444902420044
Epoch 8/20, Loss: 0.6932017803192139
Epoch 9/20, Loss: 0.669675350189209
Epoch 10/20, Loss: 0.6517990827560425
Epoch 11/20, Loss: 0.6370773911476135
Epoch 12/20, Loss: 0.6244461536407471
Epoch 13/20, Loss: 0.6133967041969299
Epoch 14/20, Loss: 0.6036434173583984
Epoch 15/20, Loss: 0.5949970483779907
Epoch 16/20, Loss: 0.587314248085022
Epoch 17/20, Loss: 0.5804778933525085
Epoch 18/20, Loss: 0.5743885040283203
Epoch 19/20, Loss: 0.5689595341682434
Epoch 20/20, Loss: 0.5641153454780579


In [12]:
with torch.no_grad():
    y_valid_pred = X_valid @ w + b
    valid_loss = ((y_valid_pred - y_valid) ** 2).mean()

    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}") #print epoch and loss

Epoch 20/20, Loss: 0.5641153454780579


# PyTorch high-level API
Replacing manual setup: nn.Linear manages w and b & the optimizer updates them

In [13]:
import torch.nn as nn  

#Build model
torch.manual_seed(42)  # to get reproducible results
model = nn.Linear(in_features=n_features, out_features=1)    

In [14]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [15]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [16]:
model(X_train[:2])

tensor([[0.8036],
        [0.4508]], grad_fn=<AddmmBackward0>)

In [17]:
#chooses the gradient descent algorithm, model params tells which weights and biases to update
#sets the update step size using learning rate
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)  

#mean squared error loss function. prediction error, measures how far predictions are from the correct answers.
mse = nn.MSELoss()

In [18]:
#batch gradient descent function, processses the training set and updates model once per epoch. 
#the inputs are moderl, optimizer, loss function (criterion), training fuatures, targets & # of epoches).

def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):     
    for epoch in range(n_epochs):           #loop repeats the training steps n-epoches times, Epoch = full pass trpugh trainings set
        y_pred = model(X_train)             #Uses the model’s current weights and biases to predict a value for every training example.
        loss = criterion(y_pred, y_train)   #Compares predictions with actual targets, calculates a single error value (mean squared error)
        loss.backward()                     #Calculates the gradients: how the loss changes with each weight and bias.No params updates yet.
        optimizer.step()                    #Updates the weights and biases usonmg gradients & learning rate
        optimizer.zero_grad()               #Clears previous gradients
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")  #loss was calculated before that epoch’s weight update.


In [19]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.279624938964844
Epoch 2/20, Loss: 0.7741653323173523
Epoch 3/20, Loss: 0.6209331154823303
Epoch 4/20, Loss: 0.6017913222312927
Epoch 5/20, Loss: 0.5913612246513367
Epoch 6/20, Loss: 0.583022952079773
Epoch 7/20, Loss: 0.5759017467498779
Epoch 8/20, Loss: 0.5696936845779419
Epoch 9/20, Loss: 0.564237117767334
Epoch 10/20, Loss: 0.559423565864563
Epoch 11/20, Loss: 0.5551694631576538
Epoch 12/20, Loss: 0.5514057278633118
Epoch 13/20, Loss: 0.5480729937553406
Epoch 14/20, Loss: 0.5451197624206543
Epoch 15/20, Loss: 0.5425010919570923
Epoch 16/20, Loss: 0.5401774644851685
Epoch 17/20, Loss: 0.5381143093109131
Epoch 18/20, Loss: 0.5362810492515564
Epoch 19/20, Loss: 0.5346510410308838
Epoch 20/20, Loss: 0.5332005620002747


In [20]:
X_new = X_test[:3]  # pretend these are new instances
with torch.no_grad():
    y_pred = model(X_new)  # use the trained model to make predictions
y_pred

tensor([[0.8269],
        [1.6917],
        [2.6476]])

Other example

### Regression MLP
Adding hidden layers to make a more flexible model

In [21]:
torch.manual_seed(42)               
model = nn.Sequential(
    nn.Linear(n_features, 50),   #1st hidden layer, with inputs (=8), output (= 50) hyperparameters
    nn.ReLU(),                   #Relu activation module in the 1st hidden layer, no model params, input = output
    nn.Linear(50, 40),           #2nd hidden layer, inputs = 50, same as previous layer, output any number (=40)
    nn.ReLU(),                   #Relu activation module implmenting activation fucntion in 2nd layer
    nn.Linear(40, 1)             #Output layer MUST have 40 inputs, ouput must match the target dimensionality = 1
)

#### Training the model

In [22]:
#Training the model
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) #optimizer controls how the model’s weights and biases are updated. The learning rate scales those updates.
mse = nn.MSELoss()                      #The loss function measures how far predictions are from the correct answers. MSE averages the squared differences.
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.994747161865234
Epoch 2/20, Loss: 2.073366165161133
Epoch 3/20, Loss: 1.0232110023498535
Epoch 4/20, Loss: 0.8789840936660767
Epoch 5/20, Loss: 0.7960251569747925
Epoch 6/20, Loss: 0.744418203830719
Epoch 7/20, Loss: 0.7113664746284485
Epoch 8/20, Loss: 0.6893779635429382
Epoch 9/20, Loss: 0.6739959716796875
Epoch 10/20, Loss: 0.6625288724899292
Epoch 11/20, Loss: 0.6534068584442139
Epoch 12/20, Loss: 0.6456424593925476
Epoch 13/20, Loss: 0.6387075781822205
Epoch 14/20, Loss: 0.6323261260986328
Epoch 15/20, Loss: 0.6263551115989685
Epoch 16/20, Loss: 0.6206678152084351
Epoch 17/20, Loss: 0.6152241826057434
Epoch 18/20, Loss: 0.6099879145622253
Epoch 19/20, Loss: 0.6049358248710632
Epoch 20/20, Loss: 0.6000460386276245


### Mini-Batch Gradient Descent
Updatew and b after a small group of examples instead of the whole training set

In [23]:
#Preparing Training Dataset and pulling batches from this dataset. 
#Provides groups of 32 examples during training. Shuffling changes their order for each pass through the data.

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)   #loader organise the data only

In [24]:
torch.manual_seed(42)
device = torch.device("cpu") #using CPU for this trainigns
model = model.to(device) #moving the model’s parameters to that device

In [25]:
#Training 
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()                                                       #setting training mode
    for epoch in range(n_epochs):                                       #loop performs learning
        total_loss = 0.
        for X_batch, y_batch in train_loader:                           #Moves a batch to the model’s device.
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)                                     #Makes predictions 
            loss = criterion(y_pred, y_batch)                           
            total_loss += loss.item()                                   #calculates the loss
            loss.backward()                                             # Calculate new gradients
            optimizer.step()                                            #Updates weights
            optimizer.zero_grad()                                       # Clears gradient for the next batch

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [26]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) #optimizer controls how the model’s weights and biases are updated. The learning rate scales those updates.
mse = nn.MSELoss()    #The loss function measures how far predictions are from the correct answers. MSE averages the squared differences.

train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 0.5371
Epoch 2/20, Loss: 0.4798
Epoch 3/20, Loss: 0.4248
Epoch 4/20, Loss: 0.3933
Epoch 5/20, Loss: 0.3734
Epoch 6/20, Loss: 0.3624
Epoch 7/20, Loss: 0.3471
Epoch 8/20, Loss: 0.3334
Epoch 9/20, Loss: 0.3276
Epoch 10/20, Loss: 0.3204
Epoch 11/20, Loss: 0.3138
Epoch 12/20, Loss: 0.3099
Epoch 13/20, Loss: 0.3099
Epoch 14/20, Loss: 0.3004
Epoch 15/20, Loss: 0.2992
Epoch 16/20, Loss: 0.2976
Epoch 17/20, Loss: 0.2949
Epoch 18/20, Loss: 0.2924
Epoch 19/20, Loss: 0.2972
Epoch 20/20, Loss: 0.2910


### Model Evaluation


In [27]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))

In [28]:
#build a TensorDataset and a DataLoader for validation set, 
#pass it to our evaluate() function to compute the validation MSE

valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

tensor(0.2877)

In [29]:
evaluate(model, valid_loader, mse,
aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))

tensor(0.5364)

In [30]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end

In [31]:
import torch
from torch import nn

device = torch.device("cpu")
model = model.to(device)

rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate_tm(model, valid_loader, rmse)

tensor(0.5368)

### NN FLOW:
**Build the model**
* model = nn.Linear(8, 1)

**Choose how to measure errors and update the model**
* mse = nn.MSELoss()  - Measure errors and update
* optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  - The optimizer controls how the model’s weights and biases are updated. The learning rate scales those updates.

**Train**
* optimizer.zero_grad() - Clear gradients left from an earlier update
* y_pred = model(X_batch) - Make predictions using the current weights
* loss = mse(y_pred, y_batch) - Measure the prediction errors
* loss.backward() - Calculate gradients: how each weight and bias affects the loss.
* optimizer.step() - Use those gradients to update the weights and biases.

This weight update is the learning. One pass through all training batches is an epoch. With 20 epochs, the pass gets repeated 20 times.

**Evaluate:**

#selects evaluation behavior for layers that behave differently during training, skips gradient calculations
    model.eval()                            
        with torch.no_grad():                   

        y_pred = model(X_valid)
        valid_mse = mse(y_pred, y_valid)

    print("Validation MSE:", valid_mse.item())


**Batch GD: one update per epoch**

    y_pred = model(X_train)

**Mini-batch GD: many updates per epoch**

    for X_batch, y_batch in train_loader:
      y_pred = model(X_batch)     


The GD variants differ in how much training data is used to calculate the gradient before each weight update.

- Batch GD: all observations → 1 update
- SGD: 1 observation → 1 update
- Mini-batch GD: small group of observations → 1 update